# [센서 실습 19] 자이로스코프 – MPU-6050 (자이로)

**MPU-6050**의 자이로 레지스터를 읽어 회전 속도(°/s)를 구하고, **적분**하여 회전 각도를 계산합니다. 정지 상태에서 **바이어스 보정**을 하지 않으면 각도가 점점 틀어지는(드리프트) 현상도 확인합니다.

**이 노트북의 흐름**
1. 센서의 원리와 부품 사양
2. 라즈베리파이 연결 방법
3. 실제 센서를 읽는 코드 (라즈베리파이용)
4. 장비가 없을 때 쓰는 **가상 센서** (같은 코드로 실행)
5. 수집 → 판정 → 시각화 → 저장·IoT 전송

> 장비가 없어도 `USE_REAL_SENSOR = False` 상태 그대로 위에서부터 `Shift + Enter`로 모두 실행됩니다.

## 1. 센서의 원리
MEMS 자이로는 진동하는 작은 구조물이 회전할 때 받는 **코리올리 힘**을 측정해 **각속도(°/s)** 를 구합니다.
- 레지스터 `0x43`부터 6바이트: X·Y·Z 각속도 (16비트 부호 정수)
- ±250°/s 범위에서 **1°/s = 131**
- 각도 = 각속도를 시간에 대해 더해 나감(**적분**): $\theta = \sum \omega \cdot \Delta t$
- 정지해 있어도 0이 아닌 작은 값(**바이어스**)이 나오며, 적분하면 오차가 계속 쌓임(**드리프트**)
  → 시작할 때 정지 상태 평균을 빼 주는 **보정**이 필요 (드론·스마트폰은 가속도계·자력계와 함께 보정)

### 부품 사양
| 항목 | 내용 |
|---|---|
| 범위 | ±250 / 500 / 1000 / 2000 °/s |
| 분해능 | 16비트 (±250°/s: 131 LSB/(°/s)) |
| 영점 오차 | ±20 °/s 이내 (개체마다 다름) |
| 통신 | I2C 주소 0x68 |

## 2. 라즈베리파이 연결 방법
**준비물**: MPU-6050 모듈(GY-521), 점퍼선 4개

| 연결할 곳 | 라즈베리파이 | 설명 |
|---|---|---|
| MPU-6050 VCC | 3.3V (1번 핀) |  |
| MPU-6050 GND | GND (6번 핀) |  |
| MPU-6050 SCL | GPIO3 / SCL (5번 핀) | I2C 클록 |
| MPU-6050 SDA | GPIO2 / SDA (3번 핀) | I2C 데이터 (주소 0x68) |



**인터페이스 활성화**
- **I2C**: `sudo raspi-config` → Interface Options → **I2C** → Enable, 연결 확인: `i2cdetect -y 1`

**라이브러리 설치 (라즈베리파이 터미널)**
```bash
pip3 install smbus2
```

## 3. 공통 준비 – 한글 폰트와 라이브러리

In [ ]:
# Colab에서 그래프의 한글이 깨지지 않도록 나눔고딕 폰트를 설정합니다. (처음 한 번 실행)
import os, subprocess
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(font_path):
    subprocess.run("apt-get -qq -y install fonts-nanum", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 설정 완료")

In [ ]:
import time, json
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 4. 실제 센서 읽기 (라즈베리파이용)
### 4-1. 원시값 → 물리량 변환식 (실제·가상 공통)
6바이트를 각속도(°/s)로 바꾸는 식입니다.

In [ ]:
GYRO_SCALE = 131.0          # ±250°/s 범위

def to_int16(hi, lo):
    v = (hi << 8) | lo
    return v - 65536 if v >= 32768 else v

def bytes_to_gyro(block):
    """[XH, XL, YH, YL, ZH, ZL] → (gx, gy, gz) °/s"""
    return tuple(round(to_int16(block[i], block[i + 1]) / GYRO_SCALE, 3) for i in (0, 2, 4))

print(bytes_to_gyro([0, 0, 0, 0, 0x2F, 0xF4]))     # 0x2FF4 = 12276 → 93.7°/s

### 4-2. 실제 센서 클래스
`0x43`부터 6바이트를 읽습니다.

> 이 셀은 **클래스를 정의만** 합니다. 하드웨어 라이브러리는 객체를 만들 때(`__init__`) 불러오므로 Colab에서 실행해도 오류가 나지 않습니다. 실제 동작은 라즈베리파이에서 `USE_REAL_SENSOR = True`로 해야 합니다.

In [ ]:
class RealMPU6050Gyro:
    PWR_MGMT_1, GYRO_XOUT_H = 0x6B, 0x43

    def __init__(self, address=0x68, bus=1):
        from smbus2 import SMBus
        self.bus, self.addr = SMBus(bus), address
        self.bus.write_byte_data(self.addr, self.PWR_MGMT_1, 0)

    def read(self):
        block = self.bus.read_i2c_block_data(self.addr, self.GYRO_XOUT_H, 6)
        gx, gy, gz = bytes_to_gyro(block)
        return {"gx": gx, "gy": gy, "gz": gz}

## 5. 장비가 없을 때 – 가상 센서
### 왜 가상 센서를 쓰나요?
실제 센서의 값은 다음 순서로 만들어집니다.

```
① 전기 신호 (전압, 펄스, 0/1)  →  ② 원시값 raw (ADC 숫자, 바이트, 문자열)  →  ③ 변환식  →  ④ 물리량 (°C, ppm, cm …)
```
가상 센서는 ①을 건너뛰고 **②원시값을 실제 센서와 똑같은 형식으로** 만들어 줍니다.
따라서 ③변환식과 그 뒤의 판정·시각화·저장 코드는 실제 장비와 가상 장비에서 **완전히 같은 코드**를 사용합니다.

```
[장비 있음]  센서 ──배선──▶ 라즈베리파이 ──▶ Real….read()  ─┐
                                                             ├──▶ 변환식 → 판정 → 그래프 → 저장/전송
[장비 없음]  가정한 상황(시나리오) ──▶ Virtual….read() ──────┘
```
나중에 장비를 연결하면 `USE_REAL_SENSOR = True` **한 줄만** 바꾸면 됩니다.

### 가상 값은 이렇게 정했습니다
| 구간 | 가정한 동작 | Z축 각속도 |
|---|---|---|
| 0 ~ 2초 | 책상 위에 정지 (보정 구간) | 0 °/s |
| 2 ~ 3초 | 회전 의자처럼 왼쪽으로 90° 회전 | 매끄럽게 최대 약 140 °/s |
| 3 ~ 6초 | 정지 | 0 |
| 6 ~ 7초 | 오른쪽으로 90° 되돌아옴 | 최대 약 −140 °/s |
| 7 ~ 10초 | 정지 | 0 |
| **바이어스** | 항상 +1.8 °/s | 저가 MEMS 센서의 일반적인 영점 오차 |
| 잡음 | ±0.3 °/s | |

In [ ]:
class VirtualMPU6050Gyro:
    BIAS = 1.8

    def __init__(self, dt=0.02, seed=19):
        self.dt, self.t = dt, 0.0
        self.rng = np.random.default_rng(seed)

    def _true_wz(self):
        t = self.t
        bump = lambda a: 90 * (np.pi / 2) * np.sin(np.pi * (t - a))     # 1초 동안 적분하면 90°
        if 2 <= t < 3: return bump(2)
        if 6 <= t < 7: return -bump(6)
        return 0.0

    def read(self):
        w = np.array([0.0, 0.0, self._true_wz()]) + self.BIAS + self.rng.normal(0, 0.3, 3)
        raw = np.clip(np.round(w * GYRO_SCALE), -32768, 32767).astype(int)
        block = []
        for v in raw:
            v &= 0xFFFF
            block += [v >> 8, v & 0xFF]
        self.t += self.dt
        gx, gy, gz = bytes_to_gyro(block)
        return {"gx": gx, "gy": gy, "gz": gz}

In [ ]:
# 가상 센서 동작 확인 – read()를 3번 호출
test = VirtualMPU6050Gyro(dt=0.02)
for _ in range(3):
    print(test.read())

## 6. 센서 선택과 데이터 수집
50Hz로 10초 동안 측정합니다.

- 실제 장비는 `time.sleep(INTERVAL)`으로 측정 간격만큼 기다립니다.
- 가상 센서는 기다리지 않고 시간을 건너뛰므로, 몇 시간·며칠 분량의 데이터도 바로 만들어집니다.

In [ ]:
USE_REAL_SENSOR = False      # ← 라즈베리파이에 센서를 연결했다면 True 로 변경
N_SAMPLES = 500               # 측정 횟수
INTERVAL = 0.02               # 측정 간격(초)

sensor = RealMPU6050Gyro() if USE_REAL_SENSOR else VirtualMPU6050Gyro(dt=INTERVAL)

records = []
for i in range(N_SAMPLES):
    value = sensor.read()
    records.append({"time_s": round(i * INTERVAL, 4), **value})
    if USE_REAL_SENSOR:
        time.sleep(INTERVAL)

df = pd.DataFrame(records)
print(f"{len(df)}개 측정 완료 (총 {N_SAMPLES * INTERVAL / 60:.1f}분 분량)")
df.head()

In [ ]:
df.describe().round(3)

## 7. 판정 – 이벤트 감지
1. 처음 2초(정지) 동안의 평균을 **바이어스**로 구해 뺍니다.
2. 보정 전·후 각속도를 각각 **적분**해 회전 각도를 구하고 비교합니다.

In [ ]:
CAL_S = 2.0
bias = df.loc[df.time_s < CAL_S, "gz"].mean()
df["gz_cal"] = df["gz"] - bias
df["angle_raw"] = (df["gz"] * INTERVAL).cumsum()
df["angle_cal"] = (df["gz_cal"] * INTERVAL).cumsum()
print(f"측정한 바이어스: {bias:.2f} °/s")
at = lambda s: df.loc[(df.time_s - s).abs().idxmin()]
print(f"4초 (90° 돌린 뒤): 보정 전 {at(4).angle_raw:6.1f}°  /  보정 후 {at(4).angle_cal:6.1f}°")
print(f"10초 (제자리)    : 보정 전 {at(9.98).angle_raw:6.1f}°  /  보정 후 {at(9.98).angle_cal:6.1f}°")
print("→ 보정하지 않으면 가만히 있어도 각도가 계속 틀어집니다(드리프트).")

## 8. 시각화

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax[0].plot(df.time_s, df.gz, color="gray", lw=1, label="원시 각속도")
ax[0].plot(df.time_s, df.gz_cal, "k-", lw=1.2, label="바이어스 보정 후")
ax[0].axvspan(0, CAL_S, color="0.93"); ax[0].text(0.1, 100, "보정 구간", fontsize=9)
ax[0].set_ylabel("Z축 각속도 (°/s)"); ax[0].legend(); ax[0].grid(alpha=0.3); ax[0].set_title("MPU-6050 자이로 – 회전 각도 계산")
ax[1].plot(df.time_s, df.angle_raw, "k--", lw=1.2, label="보정 전 (드리프트)")
ax[1].plot(df.time_s, df.angle_cal, "k-", lw=1.5, label="보정 후")
ax[1].axhline(90, color="gray", ls=":"); ax[1].axhline(0, color="gray", ls=":")
ax[1].set_ylabel("회전 각도 (°)"); ax[1].set_xlabel("시간 (초)"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 9. 저장과 IoT 전송
측정값은 CSV 파일로 저장하고, IoT 서버로 보낼 때는 보통 **JSON** 형식의 메시지를 **MQTT** 프로토콜로 전송합니다.
(Colab에서는 왼쪽 📁 메뉴에서 저장된 CSV를 내려받을 수 있습니다.)

In [ ]:
csv_name = "자이로스코프_data.csv"
df.to_csv(csv_name, index=False, encoding="utf-8-sig")
print("CSV 저장:", csv_name)

def to_jsonable(v):
    if isinstance(v, (bool, np.bool_)): return bool(v)
    if isinstance(v, (int, np.integer)): return int(v)
    if isinstance(v, (float, np.floating)): return None if np.isnan(v) else round(float(v), 3)
    return str(v)

last = df.iloc[-1].to_dict()
payload = json.dumps({"device": "rpi-01", "sensor": "MPU-6050 (자이로)",
                      "time": datetime.now().isoformat(timespec="seconds"),
                      "values": {k: to_jsonable(v) for k, v in last.items()}}, ensure_ascii=False)
print("IoT 서버로 보낼 메시지(JSON):")
print(payload)

# ▼ 실제로 MQTT 브로커에 보내려면 (pip install paho-mqtt) 주석을 해제하세요.
# import paho.mqtt.client as mqtt
# client = mqtt.Client()
# client.connect("broker.hivemq.com", 1883)          # 공개 테스트 브로커
# client.publish("myclass/rpi-01/gyro", payload)
# client.disconnect()

## 직접 해 보기
- 측정 시간을 60초(`N_SAMPLES = 3000`)로 늘리면 보정 전 각도 오차는 얼마나 커지나요?
- 보정 구간을 0.2초로 줄이면 보정 정확도가 어떻게 바뀌나요?
- 18번 가속도 센서의 기울기 각도와 자이로 각도를 섞는 '상보 필터'를 조사해 보세요.